# Shor's Algorithm with Qiskit

This notebook continues the introduction to quantum computing in `intro_to_quantum_computing_with_qiskit.ipynb`. The setup cells and QFT helpers below come from that previous notebook and are included here so you can run this notebook independently, without running the previous notebook first.

Run the setup cells, then work through the examples in order. The optional challenges contain TODOs that you must complete before running their unfinished cells.


## Setup

Install the dependencies and import the tools used throughout this notebook.


In [1]:
# Install the packages needed by this notebook. Run this once per environment.
%pip install -q qiskit==2.5.2 qiskit-aer==0.17.2 pylatexenc matplotlib==3.11.2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Imports used throughout the notebook.
from qiskit import QuantumCircuit, transpile
try:
    from qiskit_aer import AerSimulator
    simulator = AerSimulator()
    AER_AVAILABLE = True
except ImportError:
    # This fallback supports the ideal examples if Aer was not installed yet.
    from qiskit.providers.basic_provider import BasicSimulator
    simulator = BasicSimulator()
    AER_AVAILABLE = False
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from math import gcd, pi
from fractions import Fraction

print("All set! Qiskit is ready to go.")
if not AER_AVAILABLE:
    print("Aer was not found, so the ideal examples will use Qiskit's BasicSimulator.")


All set! Qiskit is ready to go.


## QFT helpers from the previous notebook

These functions implement the QFT and inverse QFT explained in the previous notebook. Shor's period-finding circuit uses the inverse QFT. They are copied here so all required helper code is available in this notebook.


In [ ]:
def build_qft(n_qubits):
    """Build a Quantum Fourier Transform (QFT) circuit on n_qubits."""

    # Create the circuit that will contain the QFT.
    circuit = QuantumCircuit(n_qubits, name="QFT")

    # Build the QFT from the highest-indexed qubit downward.
    # For 3 qubits: target = 2, then 1, then 0.
    for target in reversed(range(n_qubits)):

        # Create an equal superposition on the current target qubit.
        circuit.h(target)

        # Apply the phase corrections from each lower-indexed qubit.
        # For 3 qubits:
        #   target = 2 -> controls 0 and 1
        #   target = 1 -> control 0
        #   target = 0 -> no controls
        for control in range(target):

            # The phase rotation becomes smaller as the distance between
            # the control and target increases:
            #   adjacent qubits -> pi/2
            #   two apart       -> pi/4
            #   three apart     -> pi/8
            angle = pi / (2 ** (target - control))

            # Apply the controlled-phase rotation discussed above.
            # When both selected qubits are |1>, that component of the
            # state receives a phase factor of e^(i*angle).
            circuit.cp(angle, control, target)

    # The QFT construction leaves the qubit order reversed.
    # Swap the outer qubits to restore the conventional output order.
    # For 3 qubits, this swaps qubits 0 and 2.
    for left in range(n_qubits // 2):
        circuit.swap(left, n_qubits - left - 1)

    return circuit

def build_inverse_qft(n_qubits):
    """Build the inverse QFT on n_qubits."""
    return build_qft(n_qubits).inverse()


---
## 6. Shor's algorithm: factoring 15

### From the QFT to Shor's algorithm

We now have the key quantum tool we need. We have seen how the QFT can take periodic structure hidden across a quantum state and transform it into measurement outcomes that reveal information about that period.

Shor's algorithm turns that ability into something remarkable: **factoring integers**.

Suppose we are given a number such as

$$N=15$$

We already know that

$$15=3\times5$$

but imagine that we did not know its factors. For sufficiently large numbers, finding those factors can become extremely difficult for known classical algorithms. This matters in practice because the security of widely used public-key cryptosystems such as RSA is connected to the difficulty of factoring very large integers.

Shor's key insight is that factoring can be connected to the period of a particular repeating function.

Suppose we want to factor $N=15$. We first choose a number $a$ such that

$$1<a<N$$

and $a$ shares no factors with $N$. In practice, we can simply choose $a$ and efficiently calculate $\gcd(a,N)$ using the **[Euclidean algorithm](https://en.wikipedia.org/wiki/Euclidean_algorithm)**, even when the numbers are extremely large. We do not need to know the factors of $N$ beforehand.

If

$$\gcd(a,N)=1$$

then $a$ and $N$ share no factors and we can continue. Funny enough, if $\gcd(a,N)>1$, we have accidentally found a factor of $N$ already and do not need the quantum part of Shor's algorithm at all.

For $N=15$, we could choose

$$a=2$$

Since

$$\gcd(2,15)=1$$

we can continue with $a=2$.

We then consider the function

$$f(x)=a^x\bmod N$$

Here, "mod $N$" means we take the remainder after dividing by $N$. For example, $17\bmod5=2$ because dividing 17 by 5 leaves a remainder of 2. For $a=2$ and $N=15$, this becomes

$$f(x)=2^x\bmod15$$

Let's calculate the first few values:

$$
\begin{aligned}
2^0\bmod15 &= 1\\
2^1\bmod15 &= 2\\
2^2\bmod15 &= 4\\
2^3\bmod15 &= 8\\
2^4\bmod15 &= 1\\
2^5\bmod15 &= 2
\end{aligned}
$$

The outputs therefore repeat:

$$1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8,\ldots$$

The pattern repeats every 4 inputs, so this function has period

$$r=4$$

This is where the connection to factoring appears. If we know this period, then

$$a^r\equiv1\pmod N$$

For our example,

$$2^4\equiv1\pmod{15}$$

which means that $2^4-1$ is divisible by 15. We can therefore write

$$2^4-1=(2^2-1)(2^2+1)=3\times5$$

and the numbers 3 and 5 are exactly the factors we were looking for.

This will not always work quite this directly for every choice of $a$ and $N$, and we will develop the general procedure below. The important connection for now is

$$
\text{factoring }N
\rightarrow
\text{find the period of }a^x\bmod N
\rightarrow
\text{use that period to find factors of }N
$$

So **period finding is not an unrelated problem**. Shor deliberately chooses a periodic function whose period is mathematically connected to the factors of the number we are trying to factor.

The problem is that finding this period can itself become difficult for a classical computer when the numbers become very large. This is where the quantum part of Shor's algorithm enters.

The quantum computer evaluates the periodic function across a superposition of many input values. This creates a quantum state containing the repeating structure of the function. As we learned in the QFT section, simply measuring that state would reveal only one value rather than the overall pattern.

The Fourier-transform machinery allows that repeating structure to affect the probabilities of our measurements. From those measurement results, we can recover information about the period $r$.

The overall strategy is therefore

$$
\text{number to factor}
\rightarrow
\text{periodic function}
\rightarrow
\text{quantum period finding}
\rightarrow
\text{period}
\rightarrow
\text{factors}
$$

We will build this process for $N=15$. Because 15 is deliberately small, we can follow every step and see exactly where the quantum computation enters and how the period ultimately gives us the factors 3 and 5.

### From the period to the factors

Our example above made the final step particularly easy to see because

$$2^4-1=(2^2-1)(2^2+1)=3\times5$$

More generally, once Shor's algorithm has found a suitable even period $r$, we calculate

$$\gcd(a^{r/2}-1,N)$$

and

$$\gcd(a^{r/2}+1,N)$$

These greatest common divisors can reveal non-trivial factors of $N$.

For our example, $a=2$, $r=4$, and $N=15$, so

$$
\gcd(2^{4/2}-1,15)=\gcd(3,15)=3
$$

and

$$
\gcd(2^{4/2}+1,15)=\gcd(5,15)=5
$$

We have therefore recovered

$$15=3\times5$$

The important point is that the quantum computer does **not** directly output the factors 3 and 5. Its difficult job is to help us find the period $r$. Once we know $r$, extracting the factors is a classical calculation.

### Registers and controlled multiplication

We now know exactly what we want the quantum computer to find. For our example,

$$f(x)=2^x\bmod15$$

produces the repeating sequence

$$1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8,\ldots$$

with period $r=4$. Earlier, we saw why finding this period matters: once we know $r$, we can use classical GCD calculations to recover the factors of 15.

Our goal now is to design a quantum circuit that can extract information about this period.

But there is an immediate problem: the QFT cannot simply look at the equation $2^x\bmod15$ and tell us its period. We first need to **encode the behaviour of this function into a quantum state**. Once the quantum state contains information about the repeating pattern, the Fourier-transform techniques we developed earlier can help us extract it.

To do this, Shor's circuit uses two groups of qubits. A group of qubits used together for a particular purpose is called a **register**.

Our circuit will contain two registers:

* A **three-qubit counting register**. This is the register we will eventually apply the inverse QFT to and measure. Its job is to collect information about the period.

* A **four-qubit work register**. This register is used to calculate and store values of $2^x\bmod15$. Four qubits are enough because they can represent the integers from 0 through 15 in binary. The work register begins in $|0001\rangle=|1\rangle$ rather than $|0000\rangle=|0\rangle$ because we will be performing multiplication. Starting from 0 would always give 0 after multiplication, while starting from 1 allows us to generate the values of the function.

To see why multiplication appears here at all, remember what exponentiation actually means. A power such as

$$2^3$$

is just repeated multiplication:

$$2^3=1\times2\times2\times2$$

If we take the result modulo 15 after each multiplication, starting from 1 gives

$$
1
\rightarrow
2
\rightarrow
4
\rightarrow
8
\rightarrow
1
\rightarrow\cdots
$$

because

$$
\begin{aligned}
(1\times2)\bmod15 &= 2\\
(2\times2)\bmod15 &= 4\\
(4\times2)\bmod15 &= 8\\
(8\times2)\bmod15 &= 1
\end{aligned}
$$

These are exactly the values we calculated earlier for

$$f(x)=2^x\bmod15$$

By repeatedly multiplying by 2 and taking the result modulo 15, the circuit generates the same periodic sequence we studied earlier. **Modular multiplication therefore provides the circuit with a way to encode the values of $f(x)=2^x\bmod15$ into the quantum state**, allowing their periodic structure to be used for period finding.

Remember that our goal is to discover the **period** of the sequence

$$1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8,\ldots$$

A period describes how the function behaves across different values of $x$. Knowing only one value, such as

$$f(3)=2^3\bmod15=8$$

does not tell us where the sequence repeats. We need information about the function for multiple inputs so that its repeating structure can appear.

This is where superposition becomes useful. We can prepare the counting register in a superposition of different values of $x$, and then use those values to control the modular multiplication. This allows the quantum state to encode the relationship between many inputs $x$ and their corresponding outputs $2^x\bmod15$ at the same time.

This is where the counting register and **controlled operations** enter.

We have already encountered controlled operations with gates such as CNOT: an operation is performed only when its control qubit is $|1\rangle$. Here, instead of conditionally applying a single X gate, the counting qubits conditionally perform modular multiplication on the work register.

There is one additional idea we need. The counting register stores $x$ in binary. With three counting qubits, for example,

$$x=x_0 2^0+x_1 2^1+x_2 2^2$$

where each $x_j$ is either 0 or 1. Therefore,

$$2^x
=
2^{x_0 2^0}
2^{x_1 2^1}
2^{x_2 2^2}$$

This equation explains why the controlled operation depends on whether a counting qubit is $|0\rangle$ or $|1\rangle$. Each counting qubit represents one binary digit of $x$, and that digit determines whether its corresponding power contributes to the exponent.

For example,

$$x=3=011_2$$

means

$$x=0(4)+1(2)+1(1)=2+1$$

so

$$2^3=2^{2+1}=2^2\times2^1$$

The two $1$ bits tell us that the factors $2^2$ and $2^1$ must be included. The $0$ bit tells us that the factor $2^4$ is absent.

This gives us a direct way to build the calculation with controlled operations: **when a counting qubit is $|1\rangle$, apply the multiplication associated with that binary position to the work register. When it is $|0\rangle$, that multiplication is skipped.**

This means that each binary digit of $x$ can control one piece of the multiplication. The three counting qubits can control multiplication by

$$2^{2^0}=2,\qquad 2^{2^1}=4,\qquad 2^{2^2}=16$$

with each multiplication performed modulo 15.

Since

$$16\bmod15=1$$

the corresponding modular multipliers for this example are therefore

$$2,\qquad4,\qquad1$$

Each multiplication is applied only when its corresponding counting qubit is $|1\rangle$. Together, the three controlled operations produce the same result as multiplying by $2^x\bmod15$ for the value of $x$ represented by the counting register.

For example, suppose the counting register represents

$$x=3=011_2$$

Since

$$3=1+2$$

the operations corresponding to the $2^0$ and $2^1$ positions are activated. Starting from 1 in the work register, this gives

$$1\times2\times4\bmod15=8$$

which is exactly

$$2^3\bmod15=8$$

The crucial quantum advantage is that the counting register can be placed into a **superposition of different values of $x$**. The controlled modular multiplications can therefore correlate each possible $x$ with its corresponding value of $2^x\bmod15$ within the same quantum state.

Conceptually, the circuit is creating a state containing relationships of the form

$$
|x\rangle|1\rangle
\rightarrow
|x\rangle|2^x\bmod15\rangle
$$

across the superposition.

This is the connection we need. The repeating sequence we studied classically is now encoded into relationships between the two quantum registers.

For this small $N=15$ example, the modular multiplication operations can be constructed using gates we already know, particularly controlled-SWAP and controlled-X gates. We will build these operations explicitly rather than treating modular multiplication as a mysterious new quantum gate.

The important picture to keep in mind is

$$
\text{superposition of }x
\rightarrow
\text{controlled modular multiplication}
\rightarrow
\text{encode }2^x\bmod15
\rightarrow
\text{periodic structure in the quantum state}
$$
Through these controlled operations, the counting and work registers become correlated in a way that contains information about the periodic behaviour of $2^x\bmod15$.

We can then apply the **inverse QFT to the counting register**. Just as we saw earlier, the Fourier transform allows periodic structure encoded across a quantum state to produce particular patterns in the probabilities of measurement. Those measurements can then give us information about the period $r$.

We do **not** need to measure the work register. Its purpose is to help encode the periodic relationship that ultimately becomes detectable through the counting register.

<details>
<summary><strong>Mathematical summary: encoding 2ˣ mod 15 using the counting and work registers</strong></summary>

<br>

**Objective:**

$$
f(x)=2^x\bmod15
$$

$$
x=x_0 2^0+x_1 2^1+x_2 2^2
$$

$$
2^x
=
2^{x_0 2^0+x_1 2^1+x_2 2^2}
$$

$$
=
2^{x_0 2^0}
2^{x_1 2^1}
2^{x_2 2^2}
$$

$$
=
(2^1)^{x_0}(2^2)^{x_1}(2^4)^{x_2}
$$

$$
2^x\bmod15
=
2^{x_0}4^{x_1}16^{x_2}\bmod15
$$

$$
=
2^{x_0}4^{x_1}1^{x_2}\bmod15
$$

$$
x_j\in\{0,1\}
$$

$$
x_j=0
\Rightarrow
(2^{2^j})^{x_j}=1
$$

$$
x_j=1
\Rightarrow
(2^{2^j})^{x_j}=2^{2^j}
$$

$$
|x_2x_1x_0\rangle|1\rangle
\longrightarrow
|x_2x_1x_0\rangle|2^x\bmod15\rangle
$$

**Example:**

$$
x=3=011_2
$$

$$
x_2=0,\qquad x_1=1,\qquad x_0=1
$$

$$
2^3
=
(2^1)^1(2^2)^1(2^4)^0
$$

$$
=2\cdot4\cdot1
$$

$$
=8
$$

$$
8\bmod15=8
$$

$$
|011\rangle|0001\rangle
\longrightarrow
|011\rangle|1000\rangle
$$

</details>

Before building the complete circuit, let's look more closely at how these controlled modular multiplication operations work.

In [ ]:
def controlled_mult_2_mod15(circuit, control, work_qubits):
    """
    Apply y -> 2y mod 15 to the work register when the control qubit is |1>.
    If the control qubit is |0>, nothing happens to the work register.
    This operation corresponds to the 2^0 place of x:
        2^(2^0) = 2
    so this is the multiplication used when the least-significant
    binary digit of x is 1.
    """
    # Unpack the four work qubits.
    #
    # q0 is the least-significant bit:
    #
    #     q3 q2 q1 q0
    #      8  4  2  1
    #
    # For example:
    #
    #     |0001> = 1
    #     |0010> = 2
    #     |0100> = 4
    #     |1000> = 8
    q0, q1, q2, q3 = work_qubits

    # Multiplication by 2 modulo 15 moves our relevant states as
    #
    #     1 -> 2 -> 4 -> 8 -> 1
    #
    # or, in binary,
    #
    #     0001 -> 0010 -> 0100 -> 1000 -> 0001
    #
    # This is a cyclic shift of the four binary digits.
    #
    # The three controlled-SWAP gates below perform that shift.
    # Because they are controlled-SWAPs, the shift happens only
    # when the control qubit is |1>.
    circuit.cswap(control, q2, q3)
    circuit.cswap(control, q1, q2)
    circuit.cswap(control, q0, q1)


def controlled_mult_4_mod15(circuit, control, work_qubits):
    """
    Apply y -> 4y mod 15 to the work register when the control qubit is |1>.
    If the control qubit is |0>, nothing happens to the work register.
    This operation corresponds to the 2^1 place of x:
        2^(2^1) = 2^2 = 4
    so this is the multiplication used when the second binary
    digit of x is 1.
    """

    # Unpack the same four work qubits.
    q0, q1, q2, q3 = work_qubits

    # Multiplication by 4 modulo 15 moves our relevant states as
    #
    #     1 -> 4
    #     2 -> 8
    #     4 -> 1
    #     8 -> 2
    #
    # or, in binary,
    #
    #     0001 -> 0100
    #     0010 -> 1000
    #     0100 -> 0001
    #     1000 -> 0010
    #
    # This can be implemented by swapping q0 with q2
    # and q1 with q3.
    #
    # Again, the swaps occur only when control = |1>.
    circuit.cswap(control, q0, q2)
    circuit.cswap(control, q1, q3)


# There is no controlled_mult_16_mod15 function because
#
#     2^(2^2) = 2^4 = 16
#
# and
#
#     16 mod 15 = 1.
#
# Multiplication by 1 leaves the work register unchanged,
# so the third counting qubit requires no modular-multiplication gates.


print("Controlled modular multiplication functions are ready.")


# ============================================================
# Visualize the two controlled modular multiplication circuits
# ============================================================

# Each demonstration circuit has five qubits:
#
#     qubit 0     -> control qubit
#     qubits 1-4  -> four-qubit work register
#
# These are demonstrations of the building blocks.
# We will later use these operations inside the full Shor circuit.

control = 0
work_qubits = [1, 2, 3, 4]


# ------------------------------------------------------------
# Controlled multiplication by 2 mod 15
# ------------------------------------------------------------

mult_2_demo = QuantumCircuit(5)

controlled_mult_2_mod15(
    mult_2_demo,
    control,
    work_qubits
)

display(mult_2_demo.draw("mpl"))


# ------------------------------------------------------------
# Controlled multiplication by 4 mod 15
# ------------------------------------------------------------

mult_4_demo = QuantumCircuit(5)

controlled_mult_4_mod15(
    mult_4_demo,
    control,
    work_qubits
)

display(mult_4_demo.draw("mpl"))

Before building the full circuit, let's verify our custom multiplication operations. The work register is initialized to binary `1`. With the control set to `1`, multiplying by 2 should give `0010`, while multiplying by 4 should give `0100`.

In [ ]:
# Verify our controlled multiplication-by-2 operation.
#
# We will start the work register in |0001> = 1 and set the
# control qubit to |1>, which activates the controlled operation.
#
# We therefore expect:
#
#     2 × 1 mod 15 = 2
#
# so the work register should finish in |0010> = 2.


# Create a circuit with:
#   qubit 0    = control qubit
#   qubits 1-4 = four-qubit work register
#   4 classical bits = used to measure the work register
test = QuantumCircuit(5, 4)

# Set the control qubit to |1>.
# This activates the controlled modular multiplication.
test.x(0)

# Initialize the work register to |0001> = 1.
#
# Qubit 1 is q0, the least-significant bit of the work register,
# so flipping it from |0> to |1> stores the number 1.
test.x(1)

# Apply the controlled multiplication:
#
#     |y> -> |2y mod 15>
#
# Since the control is |1> and y = 1, this should produce y = 2.
controlled_mult_2_mod15(
    test,
    control=0,
    work_qubits=[1, 2, 3, 4]
)

# Measure only the four work qubits.
# Their values are stored in the four classical bits.
test.measure(
    [1, 2, 3, 4],
    [0, 1, 2, 3]
)

# Compile the circuit for our simulator.
compiled = transpile(test, simulator)

# Run the circuit 100 times.
result = simulator.run(
    compiled,
    shots=100
).result()

# Every measurement should give 0010, which represents 2.
print("2 * 1 mod 15 =", result.get_counts())

# Expected result:
#
#     {'0010': 100}
#
# because 2 × 1 mod 15 = 2.

In [ ]:
# Verify our controlled multiplication-by-4 operation.
#
# We again start the work register in |0001> = 1 and set the
# control qubit to |1>, activating the controlled operation.
#
# We expect:
#
#     4 × 1 mod 15 = 4
#
# so the work register should finish in |0100> = 4.

# Create a circuit with:
#   qubit 0    = control qubit
#   qubits 1-4 = four-qubit work register
#   4 classical bits = used to measure the work register
test = QuantumCircuit(5, 4)

# Set the control qubit to |1>.
# This activates the controlled modular multiplication.
test.x(0)

# Initialize the work register to |0001> = 1.
# Qubit 1 is the least-significant work bit, so we flip it to |1>.
test.x(1)

# Apply the controlled multiplication:
#
#     |y> -> |4y mod 15>
#
# Since the control is |1> and y = 1, this should produce y = 4.
controlled_mult_4_mod15(
    test,
    control=0,
    work_qubits=[1, 2, 3, 4]
)

# Measure the four work qubits into the four classical bits.
test.measure(
    [1, 2, 3, 4],
    [0, 1, 2, 3]
)

# Compile the circuit for our simulator.
compiled = transpile(test, simulator)

# Run the circuit 100 times.
result = simulator.run(
    compiled,
    shots=100
).result()

# Every measurement should give 0100, which represents 4.
print("4 * 1 mod 15 =", result.get_counts())

# Expected result:
#
#     {'0100': 100}
#
# because 4 × 1 mod 15 = 4.

These two tests act as **sanity checks** for our custom modular multiplication circuits: we give each one the known input $1$ and verify that multiplication by 2 produces $2$ (`0010`) and multiplication by 4 produces $4$ (`0100`). To verify the circuits comprehensively, we could repeat this process for every valid work-register input from $0$ through $14$ and check that each measured result equals the expected value modulo 15.

### Putting the period-finding circuit together

At this point, we have built and tested the individual operations needed to calculate $2^x\bmod15$. We now need to combine them with the counting and work registers to actually search for the period $r$.

Recall our overall goal:

$$
f(x)=2^x\bmod15
$$

$$
1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8,\ldots
$$

We already know that this sequence has period $r=4$. Our quantum circuit will now try to recover that period from the structure of the function.

The complete period-finding circuit will follow four main steps.

**1. Prepare the work register**

The work register begins in

$$
|0001\rangle=|1\rangle
$$

because our modular multiplications build the values of $2^x\bmod15$ starting from 1.

**2. Put the counting register into superposition**

The three counting qubits begin in

$$
|000\rangle
$$

Applying a Hadamard gate to each gives

$$
\frac{1}{\sqrt8}
\sum_{x=0}^{7}|x\rangle
$$

so the counting register contains all eight possible values

$$
x=0,1,2,\ldots,7.
$$

**3. Apply the controlled modular multiplications**

We now use the operations we just built.

The binary digits of $x$ determine which modular multiplications are applied:

$$
x_0=1
\Rightarrow
\times2\pmod{15}
$$

$$
x_1=1
\Rightarrow
\times4\pmod{15}
$$

$$
x_2=1
\Rightarrow
\times1\pmod{15}
$$

After these operations, the registers are correlated according to

$$
|x\rangle|1\rangle
\longrightarrow
|x\rangle|2^x\bmod15\rangle.
$$

Across the superposition, this encodes the repeating function

$$
1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8.
$$

This is exactly what our previous section was preparing us to do: the custom multiplication circuits are the building blocks that allow the full circuit to encode this periodic function.

**4. Extract the period with the inverse QFT**

At this point the periodic information exists across the quantum state, but measuring the counting register directly would not reveal the period.

We therefore apply the **inverse QFT to the counting register**. Its job is to transform the periodic structure into a pattern of measurement probabilities.

For this example, the resulting measurements concentrate at

$$
000,\qquad010,\qquad100,\qquad110
$$

corresponding to the integers

$$
0,\qquad2,\qquad4,\qquad6.
$$

These peaks contain information about the period. After we build and run the circuit, we will examine exactly how these measured values allow us to recover

$$
r=4
$$

and then use that period to recover the factors of 15.

So the circuit we are about to build has the structure

$$
\boxed{
\text{prepare registers}
\rightarrow
\text{superposition}
\rightarrow
\text{controlled modular multiplication}
\rightarrow
\text{inverse QFT}
\rightarrow
\text{measurement}
}
$$

Let's now assemble these pieces into the complete period-finding circuit.

In [ ]:
def build_inverse_qft(n_qubits):
    """
    Build the inverse QFT for n_qubits.
    build_qft(n_qubits) was defined earlier in the notebook.
    Calling .inverse() reverses that circuit.
    """
    return build_qft(n_qubits).inverse()

def shors_circuit_15():
    """
    Build the quantum period-finding circuit for:

        N = 15
        a = 2
        f(x) = 2^x mod 15

    We use:
        3 counting qubits -> represent x = 0, ..., 7
        4 work qubits     -> represent values 0, ..., 15

    The goal of the quantum circuit is to reveal the period of

        1, 2, 4, 8, 1, 2, 4, 8, ...

    which is r = 4.
    """

    # ---------------------------------------------------------
    # Create the two registers
    # ---------------------------------------------------------

    n_count = 3
    n_work = 4

    # Total:
    #   qubits 0, 1, 2    -> counting register
    #   qubits 3, 4, 5, 6 -> work register
    #
    # We also create 3 classical bits because only the
    # counting register will be measured.
    circuit = QuantumCircuit(
        n_count + n_work,
        n_count
    )

    # Store the indices of the four work qubits.
    work = list(range(n_count, n_count + n_work))


    # ---------------------------------------------------------
    # STEP 1: Prepare the counting register in superposition
    # ---------------------------------------------------------

    # Initially the counting register is |000>.
    #
    # Applying H to each qubit produces an equal superposition
    # of all 3-bit values:
    #
    #     |000>, |001>, |010>, ..., |111>
    #
    # corresponding to:
    #
    #     x = 0, 1, 2, ..., 7
    for qubit in range(n_count):
        circuit.h(qubit)


    # ---------------------------------------------------------
    # STEP 2: Initialize the work register to |0001> = 1
    # ---------------------------------------------------------

    # Every work qubit begins in |0>.
    #
    # work[0] is the least-significant work bit, so applying X
    # changes the work register from:
    #
    #     |0000> -> |0001>
    #
    # We start at 1 because the modular multiplications will
    # build the values of 2^x mod 15 from this initial value.
    circuit.x(work[0])

    # Barrier only separates the preparation stage visually. It makes each section clearly visible on the final circuit.
    # The circuit we are making is quite big. If you've gotten this far, be extremely proud of yourself, genuinely!
    # Anyway, it does not change the quantum state.
    circuit.barrier()


    # ---------------------------------------------------------
    # STEP 3: Encode f(x) = 2^x mod 15
    # ---------------------------------------------------------

    # Recall that the binary expansion of x gives:
    #
    #     x = x0(2^0) + x1(2^1) + x2(2^2)
    #
    # and therefore:
    #
    #     2^x
    #       = (2^(2^0))^x0
    #         (2^(2^1))^x1
    #         (2^(2^2))^x2
    #
    #       = 2^x0 * 4^x1 * 16^x2
    #
    # Modulo 15:
    #
    #     16 mod 15 = 1
    #
    # so the three possible controlled multiplications are:
    #
    #     counting qubit 0 -> multiply by 2 mod 15
    #     counting qubit 1 -> multiply by 4 mod 15
    #     counting qubit 2 -> multiply by 1 mod 15


    # If counting qubit 0 is |1>, multiply the work register
    # by 2 modulo 15.
    #
    # This corresponds to:
    #
    #     2^(2^0) = 2
    controlled_mult_2_mod15(
        circuit,
        control=0,
        work_qubits=work
    )

    # If counting qubit 1 is |1>, multiply the work register
    # by 4 modulo 15.
    #
    # This corresponds to:
    #
    #     2^(2^1) = 4
    controlled_mult_4_mod15(
        circuit,
        control=1,
        work_qubits=work
    )

    # Counting qubit 2 would control multiplication by:
    #
    #     2^(2^2) = 2^4 = 16
    #
    # but:
    #
    #     16 mod 15 = 1
    #
    # so this operation would simply multiply the work register
    # by 1 and leave it unchanged.
    #
    # Therefore, no gates are required for counting qubit 2.


    # At this point, the circuit has created the relationship:
    #
    #     |x>|1> -> |x>|2^x mod 15>
    #
    # across the superposition of x values.
    #
    # The periodic structure of
    #
    #     1, 2, 4, 8, 1, 2, 4, 8
    #
    # is now encoded in the combined quantum state.

    circuit.barrier()

    # ---------------------------------------------------------
    # STEP 4: Apply the inverse QFT to the counting register
    # ---------------------------------------------------------

    # The inverse QFT acts ONLY on counting qubits 0, 1, and 2.
    #
    # It converts the encoded periodic structure into a pattern
    # of measurement probabilities from which we can infer r.
    circuit.compose(
        build_inverse_qft(n_count),
        qubits=range(n_count),
        inplace=True
    )

    circuit.barrier()

    # ---------------------------------------------------------
    # STEP 5: Measure the counting register
    # ---------------------------------------------------------

    # We measure only the counting register.
    #
    # The work register has already served its purpose by helping
    # encode the periodic relationship between x and 2^x mod 15.
    circuit.measure(
        range(n_count),
        range(n_count)
    )

    return circuit


# Build the complete period-finding circuit.
shor_qc = shors_circuit_15()

# Display the circuit so we can see the complete sequence:
#
# preparation
#     -> modular multiplication
#     -> inverse QFT
#     -> measurement
shor_qc.draw("mpl", fold=40)

#### What is actually happening between the two registers?

An important detail in this circuit is that the controlled modular multiplication gates do not directly change the counting qubits $q_0,q_1,q_2$. Instead, they change the **relationship between the counting register and the work register**.

Before modular multiplication, the counting register is in a superposition while the work register always contains $1$:

$$
\frac{1}{\sqrt{8}}\sum_{x=0}^{7}|x\rangle|1\rangle
$$

After the controlled modular multiplications, each value of $x$ becomes correlated with its corresponding value of $2^x\bmod15$:

$$
\frac{1}{\sqrt{8}}\sum_{x=0}^{7}|x\rangle|2^x\bmod15\rangle
$$

For our example, this is

$$
\frac{1}{\sqrt{8}}\big(
|000\rangle|1\rangle+
|001\rangle|2\rangle+
|010\rangle|4\rangle+
|011\rangle|8\rangle+
|100\rangle|1\rangle+
|101\rangle|2\rangle+
|110\rangle|4\rangle+
|111\rangle|8\rangle
\big)
$$

Now the repeating structure becomes visible in the **correlations** between the registers:

$$
|000\rangle,\ |100\rangle
\longleftrightarrow |1\rangle
$$

$$
|001\rangle,\ |101\rangle
\longleftrightarrow |2\rangle
$$

$$
|010\rangle,\ |110\rangle
\longleftrightarrow |4\rangle
$$

$$
|011\rangle,\ |111\rangle
\longleftrightarrow |8\rangle
$$

The counting-register values associated with the same work-register value are separated by

$$
4
$$

which is exactly the period $r$.

This is the important connection between the work register and the measurement we eventually make. The work-register values group together counting states that are separated by the period. For example, $|000\rangle$ and $|100\rangle$ are both correlated with the work value $|1\rangle$, while $|001\rangle$ and $|101\rangle$ are both correlated with $|2\rangle$. Every group has the same spacing of 4.

The inverse QFT acts on the counting register while these correlations still exist. During the inverse QFT, the different counting-state amplitudes interfere with one another. For this particular periodic structure, the interference causes some possible counting-register outcomes to cancel while others reinforce each other.

After the inverse QFT, the only counting-register outcomes with non-zero probability are

$$
|000\rangle,\qquad
|010\rangle,\qquad
|100\rangle,\qquad
|110\rangle
$$

and each occurs with probability

$$
\frac14.
$$

So the sequence of events is

$$
\text{period }4
\rightarrow
\text{counting states grouped with spacing }4
\rightarrow
\text{inverse-QFT interference}
\rightarrow
000,\ 010,\ 100,\ 110
$$

This explains why we do not need to measure the four work qubits. Their role is to establish the correlations that give the counting register its periodic structure. Once those correlations have been created, the work register can remain unmeasured.

The inverse QFT transforms that periodic structure into a pattern that we can observe by measuring only the counting register.

So the key idea is

$$
\boxed{
\text{modular multiplication}
\rightarrow
\text{correlations between the registers}
\rightarrow
\text{periodic structure}
\rightarrow
QFT^{-1}
\rightarrow
\text{measurement peaks}
}
$$

The work register therefore does not need to send a value back into the counting register or be measured separately. Its interaction with the counting register has already created the structure that determines how the counting-register amplitudes interfere during the inverse QFT.

Our next task is to run the circuit, observe these measurement peaks, and understand how the measured values allow us to recover the period $r=4$.

### Run the period-finding circuit

Now let's simulate the complete circuit and check whether the inverse QFT produces the measurement pattern we predicted from the period-$4$ structure.

When we run the circuit, we measure **only the three counting qubits** $q_0,q_1,q_2$. The four work qubits are left unmeasured because their role was to create the correlations that encoded the periodic structure before the inverse QFT. Therefore, every result in our histogram will be a three-bit string such as `000`, `010`, or `110`.

Why are we using three counting qubits? Three qubits provide

$$
2^3=8
$$

possible counting states, representing $x=0$ through $7$. For this small example, that range contains two complete repetitions of our period-$4$ function:

$$
1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8
$$

which allows the inverse QFT to reveal the period cleanly.

For larger factoring problems, we generally use more counting qubits to obtain enough precision to recover the period. A common choice in Shor's algorithm is roughly **twice as many counting qubits as there are bits in $N$**. For example, if we wanted to factor

$$
N=21
$$

then

$$
21=10101_2
$$

requires 5 bits to represent, so a typical full Shor implementation might use about

$$
2(5)=10
$$

counting qubits.

Our three-qubit choice for $N=15$ is therefore a deliberately small, clean demonstration chosen because its period fits especially neatly into the counting register.

The exact bar heights vary because measurement is sampled. The important feature is the **locations of the peaks**. A clean simulation should concentrate nearly all results in `000`, `010`, `100`, and `110`, with each occurring approximately $25\%$ of the time.

After observing these peaks, we will see how the measured values can be used to recover the period $r=4$.

In [ ]:
# Run the circuit!
compiled = transpile(shor_qc, simulator)
result = simulator.run(compiled, shots=2048).result()
counts = result.get_counts()

print("Measurement results:")
print(counts)
plot_histogram(counts)

### Interpret the measurements

We have now reached the output of the quantum part of Shor's algorithm. The counting register has three qubits, so each measured bit string represents an integer $s$ from $0$ to $7$. Since

$$
2^3=8,
$$

we interpret a measurement as the fraction

$$
\frac{s}{8}.
$$

The peaks we observed were

$$
000,\qquad010,\qquad100,\qquad110,
$$

corresponding to

$$
s=0,\qquad2,\qquad4,\qquad6.
$$

Consider the measurement `010`. It represents

$$
\frac{s}{8}
=
\frac{2}{8}
=
\frac14.
$$

The denominator $4$ gives us the candidate period

$$
r=4.
$$

Similarly, measuring `110` gives

$$
\frac68=\frac34,
$$

which also has denominator $4$.

In a general run of Shor's algorithm, the measured fraction may only be **close** to a fraction whose denominator is the period. A classical method called **continued fractions** is used to find a likely small-denominator fraction from the measurement. Our $N=15$ example is especially clean, so the useful measurements reduce directly to fractions with denominator $4$.

Not every measurement gives the complete period immediately. `000` gives $0/8=0$, which provides no useful candidate, while `100` gives

$$
\frac48=\frac12,
$$

giving the candidate $r=2$. We can test a candidate period classically by checking whether

$$
2^r\bmod15=1.
$$

For $r=2$,

$$
2^2\bmod15=4\neq1,
$$

so $2$ is rejected. For $r=4$,

$$
2^4\bmod15=1,
$$

so we have successfully recovered the period

$$
\boxed{r=4}.
$$

At this point, the **quantum period-finding part is finished**. The remaining calculation is classical.

Because $r$ is even, we calculate

$$
\gcd\left(2^{r/2}-1,15\right)
\qquad\text{and}\qquad
\gcd\left(2^{r/2}+1,15\right).
$$

Substituting $r=4$ gives

$$
\gcd(2^2-1,15)=\gcd(3,15)=3
$$

and

$$
\gcd(2^2+1,15)=\gcd(5,15)=5.
$$

Therefore,

$$
\boxed{15=3\times5}.
$$

The next code cell performs this final measurement interpretation and classical factor-recovery step automatically.

In [ ]:
def interpret_results(counts, n_count, N, a):
    """Interpret the quantum measurements and recover factors of N."""

    denominator = 2 ** n_count
    factors_found = set()

    print(f"Factoring N={N} with a={a}")
    print(f"Counting-register denominator: {denominator}\n")

    # Convert each measured peak into a candidate period.
    for output_bits, count in sorted(counts.items(), key=lambda item: -item[1]):
        s = int(output_bits, 2)

        if s == 0:
            print(f"{output_bits}: {count} shots, no period information")
            continue

        fraction = Fraction(s, denominator).limit_denominator(N)
        r = fraction.denominator

        print(
            f"{output_bits}: {count} shots, "
            f"{s}/{denominator} = {fraction}, candidate r={r}"
        )

        # Check whether the candidate period can be used to extract factors.
        if r % 2 != 0:
            print("  Candidate period is odd, so skip it.")
            continue

        if pow(a, r, N) != 1:
            print("  Candidate does not satisfy the period condition, so skip it.")
            continue

        factor_1 = gcd(a ** (r // 2) - 1, N)
        factor_2 = gcd(a ** (r // 2) + 1, N)

        for factor in (factor_1, factor_2):
            if 1 < factor < N:
                factors_found.add(factor)

    # Display the final result.
    factors = sorted(factors_found)

    if len(factors) >= 2:
        print(f"\nFactors found: {factors[0]} x {factors[1]} = {N}")
    else:
        print("\nNo complete factor pair was recovered from these samples.")

    return factors


factors = interpret_results(counts, n_count=3, N=15, a=2)

### Why a useful measurement can give only part of the period

A QFT measurement gives an estimate of a fraction $s/r$. Continued fractions recovers this fraction in reduced form. If $s$ and $r$ share a factor, the reduced denominator may therefore be a **divisor of the true period**, rather than the full period. For example, $2/4 = 1/2$, so a measurement corresponding to $s=2$ and $r=4$ may produce the candidate denominator 2 even though the true period is 4.

Our helper skips candidate periods $r$ that fail the condition $a^r \bmod N = 1$. This keeps the example simple, but some skipped measurements still contain useful information.

**Optional extension:** Before rejecting a denominator $d$, test its multiples $d, 2d, 3d, \ldots$, stopping at $N$. Use the first candidate $r$ satisfying $a^r \bmod N = 1$, then apply the existing even-period and factor checks. For $a=2,\ N=15$, denominator 2 leads to candidate $r=4$; for $a=2,\ N=21$, denominator 3 leads to candidate $r=6$.

This bounded search is useful for these small teaching examples. Testing many multiples is not an efficient general solution for large integers.

### What just happened?

The quantum computer did not directly output the factors $3$ and $5$. Its job was to solve the part of the factoring problem that Shor's algorithm can accelerate: **finding the period of a modular function**.

For our example, we wanted to factor

$$
N=15
$$

and chose

$$
a=2.
$$

This gave us the periodic function

$$
f(x)=2^x\bmod15,
$$

whose values repeat as

$$
1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8,\ldots
$$

with period

$$
r=4.
$$

The quantum circuit encoded this repeating structure into correlations between the counting and work registers. The inverse QFT transformed that periodic structure into measurable peaks in the counting register. From useful measurements such as `010` and `110`, we obtained

$$
\frac{2}{8}=\frac14
\qquad\text{or}\qquad
\frac{6}{8}=\frac34,
$$

which allowed us to recover the candidate period

$$
r=4.
$$

In a general application of Shor's algorithm, the measured fraction will often only approximate a fraction related to the period, so a classical continued-fraction calculation is used to recover a candidate $r$. The candidate is then checked by verifying

$$
a^r\bmod N=1.
$$

Once we had a valid even period, the quantum computer's job was complete.

Because

$$
a^r\equiv1\pmod N,
$$

we have

$$
a^r-1\equiv0\pmod N.
$$

For an even $r$,

$$
a^r-1
=
(a^{r/2}-1)(a^{r/2}+1).
$$

This allows classical greatest-common-divisor calculations to reveal factors of $N$. In our example,

$$
\gcd(2^2-1,15)=3
$$

and

$$
\gcd(2^2+1,15)=5,
$$

giving

$$
\boxed{15=3\times5}.
$$

So Shor's algorithm is a hybrid quantum-classical algorithm. The quantum computation finds information about the period, while classical computation interprets the measurement, verifies the candidate period, and extracts the factors.

---

### The whole algorithm from beginning to end

Stepping back, the complete process we built was

$$
\boxed{
N
\rightarrow
\text{choose }a
\rightarrow
f(x)=a^x\bmod N
\rightarrow
\text{quantum period finding}
\rightarrow
r
\rightarrow
\text{classical GCDs}
\rightarrow
\text{factors of }N
}
$$

Inside the quantum period-finding step, we:

1. created a **counting register** and a **work register**,
2. placed the counting register into superposition,
3. used controlled modular multiplication to encode $a^x\bmod N$ into the work register,
4. created correlations containing the periodic structure of the function,
5. applied the inverse QFT to the counting register,
6. measured the counting register and observed peaks related to the period,
7. converted those measurements into a candidate period.

Along the way, we used many of the central ideas of quantum computing: qubits, superposition, multi-qubit states, controlled operations, entanglement and correlations, interference, measurement, the quantum Fourier transform, and hybrid quantum-classical computation.

The number $15$ was chosen because all of these ideas can be made visible using a tiny circuit. Shor's algorithm becomes important when $N$ is enormously larger.

---

### What would change with an ideal quantum computer?

Imagine that we had a perfect, large-scale quantum computer whose qubits never decohered (lost the delicate quantum state needed for computation), whose gates never made mistakes, and whose quantum states could remain intact for as long as necessary.

We could then replace our tiny

$$
N=15
$$

with a number hundreds or thousands of bits long, including the kinds of integers used by RSA.

Conceptually, the algorithm would still look remarkably familiar. We would choose an appropriate value of $a$, create large counting and work registers, compute

$$
a^x\bmod N
$$

quantum mechanically, use quantum period finding to recover $r$, and perform the final GCD calculations classically.

The enormous difference would be **scale**.

Our modular multiplication operations were small enough that we could explicitly construct them using a few controlled SWAP gates. For an RSA-sized integer, modular exponentiation would require very large reversible arithmetic circuits containing enormous numbers of quantum gates.

Our three-qubit counting register would become a much larger register capable of resolving the period with enough precision. Our four-qubit work register would become large enough to hold RSA-sized values. The inverse QFT would likewise operate over a much larger register.

We also deliberately chose an example where the period fits perfectly into our eight possible counting states. A real execution would generally produce measurements that approximate fractions related to the period, making the continued-fraction step genuinely necessary.

Still, with perfect physical qubits, these differences would primarily be engineering differences of **size and circuit complexity**. The mathematical structure you implemented in this notebook would remain recognizable.

That is the extraordinary theoretical result behind Shor's algorithm: factoring numbers that become extraordinarily difficult for known classical algorithms can be transformed into a period-finding problem that a sufficiently powerful quantum computer can solve efficiently.

---

### What would change on a realistic quantum computer?

Unfortunately, real qubits are not perfect.

Physical qubits interact with their environment, lose quantum information, and experience imperfect operations. A computation large enough to attack RSA would require an enormous number of quantum operations while preserving delicate quantum information throughout the calculation.

This is why **quantum error correction** becomes essential.

Rather than trusting one physical qubit to store one important piece of quantum information, many physical qubits can be combined to encode a much more reliable **logical qubit**. Errors can then be detected and corrected while the computation continues.

That changes the hardware picture dramatically.

A realistic RSA-scale implementation would require physical qubits for the computation itself, additional physical qubits for quantum error correction, repeated error-detection operations, fault-tolerant logical gates, additional resources for difficult operations, and a classical control system capable of decoding errors and coordinating the quantum processor in real time.

The exact resource requirements are still an active research problem, and improvements in algorithms, error-correcting codes, hardware, and computer architecture continue to change the estimates.

For perspective, a 2025 resource estimate found that RSA-2048 could theoretically be factored in less than a week using fewer than one million noisy physical qubits under its assumed hardware parameters. More recent proposed architectures have produced even lower theoretical estimates. These are resource estimates for machines that would have to satisfy demanding assumptions about error rates, operation speeds, connectivity, error correction, and control. They are not demonstrations that RSA-2048 has been factored by a quantum computer.

This gap between a circuit that works mathematically and a physical machine capable of executing it reliably is one of the central challenges of modern quantum computing.

There is good reason for optimism, however. Quantum error correction has progressed from a theoretical requirement into something researchers can experimentally build and test. Better error-correcting codes, improved physical qubits, more efficient implementations of Shor's algorithm, improved modular arithmetic, and new quantum-computer architectures can all reduce the resources required.

The challenge is enormous, but the target is not mysterious. We know the algorithm. We know why it works. We know many of the requirements for executing it. The open challenge is building a sufficiently large, sufficiently accurate, fault-tolerant quantum computer capable of carrying the computation through to completion.

---

### Q-Day

This leads to a term you may encounter frequently in quantum computing and cybersecurity: **Q-Day**.

Q-Day refers broadly to the point at which a cryptographically relevant quantum computer becomes capable of breaking important currently deployed public-key cryptography using quantum algorithms such as Shor's algorithm.

RSA is particularly relevant here because its security depends on the practical difficulty of factoring large integers. A sufficiently capable fault-tolerant quantum computer running Shor's algorithm would undermine that assumption.

Q-Day does **not** mean that every form of encryption suddenly becomes useless. Different cryptographic systems are affected differently by quantum algorithms. The immediate concern is especially important for quantum-vulnerable public-key systems such as RSA and elliptic-curve cryptography.

This is also why the world is not simply waiting for Q-Day to arrive. **Post-quantum cryptography (PQC)** uses cryptographic problems designed to remain secure against known attacks from both classical and quantum computers. Organizations and standards bodies are already working on the transition to these systems.

If you want to follow estimates and developments surrounding this milestone, **[Is it Q-Day?](https://isitqday.com/)** tracks developments in quantum computing and their possible implications for cryptography.

Any predicted date for Q-Day should be treated as an estimate rather than a known deadline. Nobody currently knows exactly when, or whether on a particular predicted schedule, hardware capable of cryptographically relevant quantum computation will be achieved.

---

### Congratulations!

You made it.

At the beginning of this notebook, concepts such as superposition, controlled gates, quantum interference, the QFT, modular exponentiation, period finding, and Shor's algorithm may have looked like separate and fairly abstract pieces of quantum computing.

You have now connected those pieces into an actual algorithm.

You built quantum states and manipulated their amplitudes. You learned how the QFT can expose periodic structure. You constructed controlled modular multiplication circuits. You used two quantum registers whose correlations encoded a mathematical function. You applied the inverse QFT, measured its output, interpreted the resulting peaks, recovered a period, and finally converted that period into the prime factors of an integer.

Factoring $15$ is a small demonstration. The ideas you used to do it are the same ideas that make Shor's algorithm one of the most important results in quantum computing.

If you made it through the notebook, ran the circuits, understood what the registers were doing, and followed how those three measured qubits ultimately led us from $15$ to $3\times5$, then you built Shor's algorithm from the ground up and understood the ideas that make it work. You should be proud.

The challenges below are completely optional. If you want to keep going, they give you a chance to modify, extend, and experiment with what you have built.

---
## 7. Further experiments

Everything below is optional. These challenges build on the period-finding algorithm you just completed and give you a chance to modify it, experiment with its behaviour, explore noise and larger circuits, and eventually extend Shor's algorithm beyond the original $N=15$ example.

---
### Challenge A: try a different base

Our implementation factored $15$ using

$$
a=2.
$$

Shor's algorithm does not require us to choose $a=2$. We can choose other values satisfying

$$
1<a<N
$$

and

$$
\gcd(a,N)=1.
$$

For this challenge, use

$$
\boxed{a=7}.
$$

Start classically. Calculate

$$
7^x\bmod15
$$

for several consecutive values of $x$ and determine its period.

Fill in the table:

| $x$ | $7^x\bmod15$ |
|:---:|:---:|
| 0 | |
| 1 | |
| 2 | |
| 3 | |
| 4 | |
| 5 | |

What period $r$ do you observe?

Once you know the period, remember how modular exponentiation was controlled by the binary digits of $x$:

$$
7^x
=
(7^{2^0})^{x_0}
(7^{2^1})^{x_1}
(7^{2^2})^{x_2}.
$$

Calculate the three values

$$
7^{2^0}\bmod15,\qquad
7^{2^1}\bmod15,\qquad
7^{2^2}\bmod15.
$$

These tell you which modular-multiplication operations the three counting qubits need to control.

In [ ]:
# Challenge A.1
# Explore the function 7^x mod 15.

N = 15
a = 7

for x in range(8):
    # TODO: calculate a^x mod N
    value = ...
    print(f"x={x}: {value}")

Now build the two modular-multiplication operations required for $a=7$.

You do **not** need to design arbitrary modular arithmetic. We only need the transformations that occur in this particular $N=15$ example.

Use the cells below to experiment with SWAP and X gates until the work register undergoes the required permutations.

You may find it useful to write the relevant decimal values in 4-bit binary first and look for a pattern in how the bits move.

In [ ]:
def controlled_mult_7_mod15(circuit, control, work_qubits):
    """
    Apply |y> -> |7y mod 15> when the control qubit is |1>.
    """

    q0, q1, q2, q3 = work_qubits

    # TODO:
    # Build the controlled multiplication-by-7 operation.
    pass


def controlled_mult_4_mod15_a7(circuit, control, work_qubits):
    """
    Apply |y> -> |4y mod 15> when the control qubit is |1>.
    """

    q0, q1, q2, q3 = work_qubits

    # TODO:
    # Build the controlled multiplication-by-4 operation.
    pass

Finally, modify the period-finding circuit.

The register sizes, initialization, inverse QFT, and measurement can remain the same as in the $a=2$ circuit. The part that must change is the controlled modular exponentiation.

After completing the circuit, run it and inspect its histogram.

**Questions to answer:**

- What measurement peaks do you observe?
- What period do they indicate?
- Do you still recover the factors $3$ and $5$?

In [ ]:
def shors_circuit_15_a7():
    n_count = 3
    n_work = 4

    circuit = QuantumCircuit(n_count + n_work, n_count)
    work = list(range(n_count, n_count + n_work))

    # Prepare the counting register.
    for qubit in range(n_count):
        circuit.h(qubit)

    # Prepare the work register in |1>.
    circuit.x(work[0])

    # TODO:
    # Apply the controlled modular multiplications required for a=7.


    # Apply the inverse QFT.
    circuit.compose(
        build_inverse_qft(n_count),
        qubits=range(n_count),
        inplace=True
    )

    # Measure only the counting register.
    circuit.measure(range(n_count), range(n_count))

    return circuit


# TODO: build, simulate, and plot your circuit.

shor_a7 = ...
counts_a7 = ...

print(counts_a7)
# plot_histogram(...)

---

### Challenge B: use more counting qubits

Our original circuit used three counting qubits, giving

$$
2^3=8
$$

possible measurement values.

What happens if we use more?

With $n$ counting qubits, a measurement $s$ represents the fraction

$$
\frac{s}{2^n}.
$$

Increasing $n$ gives the quantum period-finding procedure more possible fractions and therefore greater precision. This becomes especially important when the period does not fit neatly into a power-of-two-sized counting register.

Our period-$4$ example is unusually clean because $4$ divides $2^n$ for every $n\ge2$. That makes it a useful place to observe how the **same period appears at different register sizes**.

Complete the function below so that it works for any reasonable number of counting qubits.

For $a=2$,

$$
2^{2^j}\bmod15
$$

tells you which modular multiplication is controlled by counting qubit $j$.

You already have the multiplication-by-2 and multiplication-by-4 functions from the main notebook.

If the required multiplier is $1$, no operation is necessary.

In [ ]:
def shors_circuit_15_flexible(n_count):
    """
    Build the a=2, N=15 period-finding circuit
    using n_count counting qubits.
    """

    n_work = 4
    circuit = QuantumCircuit(n_count + n_work, n_count)
    work = list(range(n_count, n_count + n_work))

    # Prepare counting register.
    for qubit in range(n_count):
        circuit.h(qubit)

    # Prepare work register in |1>.
    circuit.x(work[0])

    # Build controlled modular exponentiation.
    for qubit in range(n_count):

        power = pow(2, 2 ** qubit, 15)

        # TODO:
        # power tells you which multiplication operation
        # this counting qubit should control.
        #
        # Handle power == 2
        # Handle power == 4
        # Handle power == 1


    # TODO: apply the inverse QFT.


    # TODO: measure the counting register.


    return circuit

Run your function using

$$
n=3,\qquad n=4,\qquad n=5.
$$

Compare the histograms.

Before running the cell, predict where the period-$4$ peaks should appear.

For example, with four counting qubits there are

$$
2^4=16
$$

possible measured integers. If the peaks represent the fractions

$$
0,\quad\frac14,\quad\frac24,\quad\frac34,
$$

which four integers should appear?

In [ ]:
for n_count in [3, 4, 5]:

    # TODO: build the circuit.
    circuit = ...

    # TODO: simulate it.
    counts = ...

    print(f"{n_count} counting qubits:")
    print(counts)

    # TODO: display its histogram.

**Think about your results:**

Did increasing the number of counting qubits change the period?

Did the number of major peaks change?

How did the integer locations of those peaks change as $2^{n}$ became larger?

Why would the extra precision become much more important for a period that does **not** divide $2^n$ exactly?

---

### Challenge C: what happens when the gates are noisy?

Everything so far has used an ideal simulator. Every gate behaved exactly as requested.

Physical quantum computers experience errors. One simple way to explore their effect is **depolarizing noise**, where a gate has some probability of introducing a random quantum error.

The helper function below builds a noisy simulator for you. You do not need to understand the implementation of the noise model for this challenge.

Your task is experimental:

1. run the same Shor circuit at several error rates,
2. compare the resulting histograms,
3. measure how much probability remains concentrated at the expected period-$4 peaks.

For our three-qubit $a=2$ circuit, the ideal peak locations are

$$
000,\qquad010,\qquad100,\qquad110.
$$

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

def run_with_noise(circuit, error_rate=0.01, shots=2048):
    """
    Run a circuit using a simple depolarizing gate-noise model.
    """

    noise_model = NoiseModel()

    noise_model.add_all_qubit_quantum_error(
        depolarizing_error(error_rate, 1),
        ["h", "x"]
    )

    noise_model.add_all_qubit_quantum_error(
        depolarizing_error(min(0.99, 2 * error_rate), 2),
        ["cx"]
    )

    noisy_simulator = AerSimulator(noise_model=noise_model)

    compiled = transpile(circuit, noisy_simulator)

    return noisy_simulator.run(
        compiled,
        shots=shots
    ).result().get_counts()

Start with an error rate of $0.1\%$, then try $1\%$, $5\%$, and $10\%$.

For each run, look at how clearly you can still identify the four expected peaks.

In [ ]:
error_rates = [0.001, 0.01, 0.05, 0.10]

for rate in error_rates:

    # TODO: run shor_qc using run_with_noise().
    noisy_counts = ...

    print(f"Error rate: {rate:.1%}")
    print(noisy_counts)

    # TODO: plot the histogram.

Now quantify what you saw.

Calculate the fraction of all shots that landed on one of the four ideal peak locations:

$$
000,\qquad010,\qquad100,\qquad110.
$$

Then compare this quantity as the error rate increases.

In [ ]:
expected_peaks = {"000", "010", "100", "110"}

for rate in error_rates:

    noisy_counts = run_with_noise(
        shor_qc,
        error_rate=rate,
        shots=4096
    )

    total_shots = sum(noisy_counts.values())

    # TODO:
    # Count how many shots landed at one of the expected peaks.
    peak_shots = ...

    peak_fraction = peak_shots / total_shots

    print(
        f"Error rate {rate:.1%}: "
        f"{peak_fraction:.1%} of measurements at expected peaks"
    )

**Think about your results:**

What happens to the histogram as the error rate increases?

At what point does the period-$4 pattern become difficult to distinguish?

Try increasing the number of shots. Does taking more measurements undo the effect of noisy gates? Why might collecting more samples help with statistical variation while still being unable to repair errors that occurred during the computation?

---

### Challenge D: put the pieces together

In the main notebook, we manually chose

$$
a=2
$$

before running Shor's algorithm.

A more complete implementation should be able to choose a base, perform some classical checks, run quantum period finding when necessary, interpret the measurement, and return factors.

For this final challenge, build a small automated factoring pipeline for

$$
N=15.
$$

Your program should:

1. randomly choose a candidate $a$ with $1<a<15$,
2. calculate $\gcd(a,15)$,
3. immediately return a factor if that GCD is already non-trivial,
4. otherwise check whether your implementation supports that base,
5. run the appropriate quantum period-finding circuit,
6. use `interpret_results()` to search the measurements for factors,
7. try again if the attempt does not produce a complete factor pair.

There is an important simplification in this challenge: **you only constructed quantum modular-multiplication circuits for a small number of bases.** Your program therefore does not need to implement arbitrary modular arithmetic.

For bases your quantum circuits do not support, simply choose another base.

This mirrors the overall structure of Shor's algorithm while keeping the quantum arithmetic within what we developed in this notebook.

In [ ]:
import random

def full_shors_15(max_attempts=10):
    """
    Attempt to factor 15 using classical preprocessing
    and the quantum period-finding circuits developed
    in this notebook.
    """

    N = 15

    for attempt in range(1, max_attempts + 1):

        # TODO 1:
        # Randomly choose an integer a with 1 < a < N.
        a = ...

        print(f"\nAttempt {attempt}: a={a}")

        # TODO 2:
        # Compute gcd(a, N).
        common_factor = ...

        # TODO 3:
        # If the GCD already gives a non-trivial factor,
        # return the factor pair immediately.


        # TODO 4:
        # Select the appropriate quantum circuit if this
        # notebook supports the chosen value of a.
        #
        # We have a circuit for a=2.
        # If you completed Challenge A, you also have a=7.
        #
        # For unsupported bases, continue to another attempt.

        circuit = ...


        # TODO 5:
        # Run the quantum circuit and collect its counts.
        counts = ...


        # TODO 6:
        # Interpret the measurements using interpret_results().
        factors = ...


        # TODO 7:
        # If two factors were recovered, return them.


    print("No factor pair was recovered.")
    return None


result = full_shors_15()
print("\nFinal result:", result)

If your pipeline successfully returns

$$
(3,5),
$$

you have connected the major pieces of Shor's algorithm into one program:

$$
\boxed{
\text{choose }a
\rightarrow
\text{classical GCD check}
\rightarrow
\text{quantum period finding}
\rightarrow
\text{interpret measurement}
\rightarrow
\text{recover factors}
}
$$

There is still an enormous jump between this educational implementation and a general implementation capable of factoring arbitrary integers. In particular, a general implementation needs reversible modular arithmetic capable of constructing $a^x\bmod N$ for arbitrary supported values of $a$ and $N$.

But that complexity lives inside the same overall algorithmic structure you have already built here.

---

### Challenge E: factor a new number

Everything so far has revolved around

$$
N=15.
$$

For this final challenge, you will take the ideas from the notebook and apply them to a number for which we have not already constructed the period-finding circuit:

$$
\boxed{N=21}
$$

Use

$$
\boxed{a=2}.
$$

This time, most of the circuit will not be provided for you.

However, you will **not** be asked to invent a general reversible modular-multiplication circuit from elementary gates. That is a substantial topic of its own. A helper for constructing those operations is provided below.

Everything else follows ideas you have already used in this notebook.

If you understood the previous sections, you have already encountered every major step you need.

#### Step 1: investigate the classical function

Before building a quantum circuit, investigate

$$
f(x)=2^x\bmod21.
$$

Calculate enough consecutive values to identify where the sequence begins repeating.

From your results, determine the period

$$
r.
$$

Do this **before** constructing the quantum circuit. Knowing the answer classically will allow you to check whether your quantum circuit works.

In [ ]:
N = 21
a = 2

# TODO:
# Print enough values of 2^x mod 21 to identify the period.

for x in range(12):
    value = ...
    print(f"x={x}: {value}")

# From the output above:
expected_period = ...

#### Step 2: choose the register sizes

The work register must be able to represent the values used modulo $21$.

Find the smallest $n_{\text{work}}$ satisfying

$$
2^{n_{\text{work}}}>21.
$$

For the counting register, use

$$
n_{\text{count}}=8.
$$

This gives

$$
2^8=256
$$

possible counting-register outcomes.

There is an important difference from our $N=15$ example. Your period for $N=21$ does **not** divide $256$ exactly. Therefore, you should not expect the inverse QFT to produce perfectly positioned peaks corresponding to exact fractions.

This is precisely the situation in which the continued-fraction interpretation becomes useful.

In [ ]:
N = 21
a = 2

n_count = 8

# TODO:
# Determine the minimum number of work qubits required.
n_work = ...

print("Counting qubits:", n_count)
print("Work qubits:", n_work)
print("Counting states:", 2 ** n_count)
print("Work-register states:", 2 ** n_work)

#### Step 3: determine the controlled multiplications

Recall that if

$$
x=x_0 2^0+x_1 2^1+\cdots,
$$

then

$$
a^x
=
\left(a^{2^0}\right)^{x_0}
\left(a^{2^1}\right)^{x_1}
\left(a^{2^2}\right)^{x_2}
\cdots
$$

Therefore, counting qubit $j$ must control multiplication by

$$
a^{2^j}\bmod N.
$$

For our new problem, calculate

$$
2^{2^j}\bmod21
$$

for every counting qubit.

Look carefully at the resulting sequence. Some multipliers may repeat.

In [ ]:
multipliers = []

for j in range(n_count):

    # TODO:
    # Calculate the modular multiplier controlled by
    # counting qubit j.
    multiplier = ...

    multipliers.append(multiplier)

    print(
        f"Counting qubit q{j}: "
        f"multiply work register by {multiplier} mod {N}"
    )

#### Step 4: construct modular multiplication

For $N=15$, we were lucky: multiplication by $2$ and $4$ modulo $15$ could be represented using simple rotations of the four work bits.

That shortcut does not generally exist.

The helper below constructs a reversible gate implementing

$$
|y\rangle
\longrightarrow
|my\bmod N\rangle
$$

for the valid modular values $0\le y<N$.

States outside that range are left unchanged so that the complete operation remains a valid permutation of the computational basis.

You do not need to modify this function.

Your job is to use it correctly when you construct modular exponentiation.

In [ ]:
from qiskit.circuit.library import UnitaryGate
import numpy as np

def modular_multiplication_gate(multiplier, N, n_work):
    """
    Construct a reversible multiplication-by-multiplier mod N gate.

    Valid values 0 <= y < N are mapped to multiplier*y mod N.
    Computational-basis states >= N are left unchanged.
    """

    dimension = 2 ** n_work
    matrix = np.zeros((dimension, dimension), dtype=complex)

    for y in range(dimension):

        if y < N:
            new_y = (multiplier * y) % N
        else:
            new_y = y

        matrix[new_y, y] = 1

    gate = UnitaryGate(
        matrix,
        label=f"×{multiplier} mod {N}"
    )

    return gate

Before using the helper, think about one requirement.

For multiplication modulo $N$ to act as a reversible permutation, the multiplier must be coprime with $N$:

$$
\gcd(m,N)=1.
$$

Check that the multipliers you found satisfy this condition.

Then test one of your multiplication gates on a known input before trusting it inside the complete Shor circuit.

Choose an input $y$, calculate

$$
my\bmod21
$$

yourself, and check whether the quantum circuit produces the same result.

In [ ]:
# TODO:
# Choose one multiplier from your list and one test input.

m = ...
test_input = ...

expected_output = (m * test_input) % N

print("Multiplier:", m)
print("Input:", test_input)
print("Expected output:", expected_output)

# Construct the gate.
mult_gate = modular_multiplication_gate(m, N, n_work)

# OPTIONAL:
# Build a small test circuit and verify that the gate
# performs the transformation you predicted.

#### Step 5: build modular exponentiation

You now know that counting qubit $j$ should control multiplication by

$$
2^{2^j}\bmod21.
$$

The helper below takes a modular-multiplication gate and turns it into a controlled operation.

Complete the loop so that **every counting qubit controls the correct multiplication**.

Remember the structure from the $N=15$ circuit:

$$
|x\rangle|1\rangle
\longrightarrow
|x\rangle|2^x\bmod N\rangle.
$$

In [ ]:
def apply_modular_exponentiation(circuit, counting, work, a, N):
    """
    Apply the controlled operations required to encode a^x mod N.
    """

    n_work = len(work)

    for j, control_qubit in enumerate(counting):

        # TODO:
        # Determine the multiplier associated with this
        # binary digit of x.
        multiplier = ...

        # Build multiplication by that value modulo N.
        mult_gate = modular_multiplication_gate(
            multiplier,
            N,
            n_work
        )

        # Turn it into a controlled operation.
        controlled_gate = mult_gate.control(1)

        # TODO:
        # Append the controlled gate.
        # It acts on:
        #   1. control_qubit
        #   2. every qubit in the work register

#### Step 6: assemble the period-finding circuit

You now have all the pieces.

Build the complete circuit yourself.

Your circuit should perform the same conceptual sequence as the $N=15$ circuit:

$$
\boxed{
\text{initialize}
\rightarrow
\text{superposition}
\rightarrow
\text{modular exponentiation}
\rightarrow
QFT^{-1}
\rightarrow
\text{measurement}
}
$$

Use the functions already developed in the notebook whenever appropriate.

Only the **counting register** should be measured.

In [ ]:
def shors_circuit_21():
    N = 21
    a = 2

    n_count = 8
    n_work = ...

    circuit = QuantumCircuit(
        n_count + n_work,
        n_count
    )

    counting = list(range(n_count))
    work = list(range(n_count, n_count + n_work))

    # TODO 1:
    # Put the counting register into equal superposition.


    # TODO 2:
    # Initialize the work register to |1>.


    # TODO 3:
    # Apply modular exponentiation using the helper
    # you completed above.


    # TODO 4:
    # Apply the inverse QFT to the counting register.


    # TODO 5:
    # Measure only the counting register.


    return circuit


shor_21 = shors_circuit_21()

shor_21.draw("mpl", fold=40)

#### Step 7: run it

You already know how to compile and simulate a Qiskit circuit.

Run your new circuit using several thousand shots and plot the measurement histogram.

Before interpreting individual measurements, look at the overall shape.

You found the period classically in Step 1. Does the histogram contain a pattern consistent with that period?

Remember that this example will not be as perfectly clean as the $N=15$ histogram. The ideal peak locations do not all correspond to exact integers when the period does not divide $2^{n_{\text{count}}}$.

In [ ]:
# TODO:
# Transpile shor_21 for the simulator.

compiled_21 = ...

# TODO:
# Run the circuit. Try 4096 shots.

result_21 = ...

# TODO:
# Extract the counts.

counts_21 = ...

print("Measurement results:")
print(counts_21)

# TODO:
# Plot the histogram.

#### Step 8: recover the period from the quantum measurements

Now comes the real test.

For eight counting qubits, a measured integer $s$ represents

$$
\frac{s}{256}.
$$

Unlike our $N=15$ example, useful measurements will generally not reduce immediately to an exact fraction whose denominator is the period.

Use the `interpret_results()` function from the main notebook as a starting point, but inspect what it produces carefully.

For one of the largest non-zero peaks:

1. convert its bit string to the integer $s$,
2. calculate $s/256$,
3. use `Fraction(...).limit_denominator(N)` to find a nearby simple fraction,
4. inspect its denominator as a candidate for $r$,
5. verify any candidate using

$$
2^r\bmod21=1.
$$

If one measurement gives only part of the information you need, inspect another major peak.

Do not use the period you calculated in Step 1 as the answer here. That value is your **check**. The goal is to see whether you can recover it from the simulated quantum measurements.

In [ ]:
from fractions import Fraction

# TODO:
# Choose one of the major non-zero measurement outcomes
# produced by YOUR quantum circuit.

measured_bits = ...

s = int(measured_bits, 2)
measured_fraction = s / (2 ** n_count)

approximation = Fraction(
    s,
    2 ** n_count
).limit_denominator(N)

print("Measured bits:", measured_bits)
print("Measured integer:", s)
print("Measured fraction:", measured_fraction)
print("Nearby simple fraction:", approximation)
print("Candidate denominator:", approximation.denominator)

# TODO:
# Check whether the candidate denominator is actually
# a period of 2^x mod 21.

#### Step 9: finish the factorization

Once you have recovered a valid period $r$, finish exactly as you did in the main notebook.

First determine whether the period is suitable for factor extraction.

Then calculate

$$
\gcd\left(2^{r/2}-1,21\right)
$$

and

$$
\gcd\left(2^{r/2}+1,21\right).
$$

If everything worked, these calculations should reveal the two non-trivial factors of $21$.

In [ ]:
# TODO:
# Enter the period recovered from the quantum measurements.

r = ...

# Verify it.
print("Period check:", pow(a, r, N))

# TODO:
# Calculate the two GCDs used by Shor's algorithm.

factor_1 = ...
factor_2 = ...

print("Recovered factors:", factor_1, "and", factor_2)
print(
    "Verification:",
    factor_1,
    "*",
    factor_2,
    "=",
    factor_1 * factor_2
)

#### You just generalized the algorithm

If you completed this challenge successfully, compare what you did with the original $N=15$ example.

You were given a new integer and had to determine:

- the behaviour and period of $a^x\bmod N$,
- the required register sizes,
- the modular multipliers associated with each counting qubit,
- how those multipliers fit into controlled modular exponentiation,
- how to assemble the complete period-finding circuit,
- how to recognize approximate QFT peaks,
- how to recover and verify a candidate period,
- and how to turn that period into factors.

The modular-multiplication helper saved you from having to design arbitrary reversible arithmetic at the gate level. Everything surrounding it, however, is the same period-finding structure developed throughout this notebook.

If you could complete this challenge without copying the $N=15$ circuit and simply changing numbers until something worked, you have more than understood the core structure of the algorithm.

---
## Where to go next

You have now gone from manipulating individual qubits to building a complete quantum algorithm. Along the way, you worked with superposition, measurement, single- and multi-qubit gates, entanglement, interference, the quantum Fourier transform, modular arithmetic, quantum period finding, and the classical post-processing that turns Shor's period into factors.

There is still much more to quantum computing, and there are several directions you can take from here.

### Strengthen the fundamentals

Try modifying circuits you already understand. Change an input state, remove a gate, add another qubit, or change the number of measurement shots. Before running the circuit, **predict what you think will happen** and then use the simulator to test your reasoning.

Being able to predict a circuit before running it is one of the best signs that you are developing a useful intuition for quantum computation.

### Learn more quantum algorithms

Shor's algorithm is only one example of how quantum mechanics can be used computationally. Other important algorithms and techniques to explore include:

- **Grover's algorithm**, which demonstrates quantum search and amplitude amplification,
- **quantum phase estimation**, which generalizes ideas that appeared in our period-finding circuit,
- **variational quantum algorithms**, which combine parameterized quantum circuits with classical optimization,
- and **quantum simulation**, one of the original motivations for building quantum computers.

### Go deeper into the mathematics

If you want to understand why quantum algorithms work at a deeper level, continue developing your knowledge of **linear algebra, complex numbers, probability, and Fourier analysis**.

Concepts such as eigenvalues and eigenvectors, tensor products, unitary matrices, inner products, and changes of basis appear throughout quantum computing. The circuit diagrams eventually become another way of expressing this mathematics.

### Keep learning Qiskit

The [IBM Quantum Learning platform](https://learning.quantum.ibm.com/) contains courses, tutorials, and exercises that continue beyond the material covered here.

As you become more comfortable with Qiskit, you can move beyond ideal simulation and explore transpilation, realistic noise, quantum error correction, and execution on available quantum hardware.

### Most importantly: experiment

Do not feel that you need to understand all of quantum computing before building things.

Take circuits from this notebook and break them. Change the gates. Change the initial states. Add measurements. Increase the register sizes. Try different parameters. Predict what should happen, run the experiment, and investigate whenever the result surprises you.

You now have enough background to read a quantum circuit and ask the most important questions:

**What state am I preparing? What transformation am I applying? Why am I applying it? What information will measurement give me?**

Keep asking those questions, and the more advanced material has somewhere to attach.

---